In [0]:
# Databricks notebook source
# ══════════════════════════════════════
# 00_CONFIG — Configuracoes do Projeto
# Squad 3 — Arquitetura Medalhao
# Batch Lojas Fisicas
# ══════════════════════════════════════



import os
from pathlib import Path
from datetime import datetime
import pytz

In [0]:
# Carrega variaveis do arquivo .env

def load_env_file(env_path="../.env"):
    """
    Carrega variaveis do arquivo .env sem usar python-dotenv.
    Lanca erro claro se o arquivo nao for encontrado.
    """
    env_file = Path(env_path)

    if not env_file.exists():
        raise FileNotFoundError(
            f"Arquivo .env nao encontrado em: {env_file.resolve()}"
        )

    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()

        if not line or line.startswith("#"):
            continue

        if "=" not in line:
            continue

        key, value = line.split("=", 1)
        os.environ[key.strip()] = value.strip().strip('"').strip("'")


def get_required_env(key):
    """
    Retorna valor de variavel de ambiente.
    Lanca erro explicito se ausente ou vazia.
    """
    value = os.getenv(key)

    if value is None or value.strip() == "":
        raise ValueError(
            f"Variavel de ambiente ausente ou vazia: {key}. "
            f"Verifique o arquivo .env"
        )

    return value

print("Funcoes load_env_file e get_required_env criadas!")

In [0]:
# Carrega variaveis do arquivo .env

load_env_file("../.env")

# ADLS Gen2
ADLS_CLIENT_ID            = get_required_env("client_id")
ADLS_TENANT_ID            = get_required_env("tenant_id")
ADLS_CLIENT_SECRET        = get_required_env("client_secret")
ADLS_STORAGE_ACCOUNT_NAME = get_required_env("storage_account_name")

# SQL Server
SQL_HOST     = get_required_env("jdbc_hostname")
SQL_DATABASE = get_required_env("jdbc_database")
SQL_USERNAME = get_required_env("jdbc_username")
SQL_PASSWORD = get_required_env("jdbc_password")
SQL_PORT     = "1433"

# Schema SQL Server
TARGET_SCHEMA = get_required_env("sql_schema")

print("Variaveis carregadas com sucesso!")
print(f"   ADLS_STORAGE_ACCOUNT_NAME : {ADLS_STORAGE_ACCOUNT_NAME}")
print(f"   TARGET_SCHEMA             : {TARGET_SCHEMA}")
print(f"   SQL_HOST                  : {SQL_HOST}")

In [0]:
# Caminhos

# caminhos padronizados no ADLS

RAW_CONTAINER = get_required_env("container_name")

RAW_ROOT_PATH = (
    f"abfss://{RAW_CONTAINER}"
    f"@{ADLS_STORAGE_ACCOUNT_NAME}.dfs.core.windows.net/"
)

BATCH_DATA_PATH  = get_required_env("batch_data_path")
RAW_BATCH_PATH   = f"{RAW_ROOT_PATH}{BATCH_DATA_PATH}/"
BRONZE_BASE_PATH = f"{RAW_ROOT_PATH}{get_required_env('bronze_path')}/"
SILVER_BASE_PATH = f"{RAW_ROOT_PATH}{get_required_env('silver_path')}/"
GOLD_BASE_PATH   = f"{RAW_ROOT_PATH}{get_required_env('gold_path')}/"

print("Caminhos configurados:")
print(f"   RAW_BATCH_PATH   : {RAW_BATCH_PATH}")
print(f"   BRONZE_BASE_PATH : {BRONZE_BASE_PATH}")
print(f"   SILVER_BASE_PATH : {SILVER_BASE_PATH}")
print(f"   GOLD_BASE_PATH   : {GOLD_BASE_PATH}")

In [0]:
# Metadados

def get_dt_brasilia():
    """Retorna datetime atual no fuso de Brasilia como string."""
    fuso  = pytz.timezone("America/Sao_Paulo")
    agora = datetime.now(fuso)
    return agora.strftime("%Y-%m-%d %H:%M:%S")


def get_bronze_ingested_at():
    """Retorna datetime de ingestao na Bronze no fuso de Brasilia."""
    return get_dt_brasilia()


def get_silver_processed_at():
    """Retorna datetime de processamento na Silver no fuso de Brasilia."""
    return get_dt_brasilia()

print("Funcoes de metadado criadas!")
print(f"   bronze_ingested_at  : {get_bronze_ingested_at()}")
print(f"   silver_processed_at : {get_silver_processed_at()}")

In [0]:
#  

print("=" * 55)
print("00_config carregado com sucesso!")
print("=" * 55)
print(f"""
Variaveis disponiveis:
   ADLS_CLIENT_ID            : {ADLS_CLIENT_ID[:8]}...
   ADLS_STORAGE_ACCOUNT_NAME : {ADLS_STORAGE_ACCOUNT_NAME}
   RAW_CONTAINER             : {RAW_CONTAINER}
   RAW_BATCH_PATH            : {RAW_BATCH_PATH}
   BRONZE_BASE_PATH          : {BRONZE_BASE_PATH}
   SILVER_BASE_PATH          : {SILVER_BASE_PATH}
   GOLD_BASE_PATH            : {GOLD_BASE_PATH}
   TARGET_SCHEMA             : {TARGET_SCHEMA}
   SQL_HOST                  : {SQL_HOST}
""")